# Tax

Taxes are percentage-based charges applied on top of energy costs. The `Tax` class lets you define
tax rates that are applied to the structured cost output produced by a `Tariff` or `Contract`.

A `Tax` is a versioned collection of rules, each containing:
- a `default` rate — the fallback percentage applied to any cost not matched by a more specific rule
- an optional list of `rates`, each targeting specific cost columns via a pattern

The result is a single `(taxes, total, total)` column summing all applied tax amounts.

### 1. A flat rate

The simplest tax applies a single percentage to all costs. Create a `Tax` with one version and a `default` rate:

In [1]:
import datetime as dt

import pandas as pd

from energy_cost.tax import Tax, TaxVersion

tax = Tax(versions=[TaxVersion(start=dt.datetime(2020, 1, 1, tzinfo=dt.UTC), default=0.21)])
tax

Tax(versions=[TaxVersion(start=datetime.datetime(2020, 1, 1, 0, 0, tzinfo=datetime.timezone.utc), default=0.21, rates=[])])

`Tax.apply()` receives the DataFrame produced by `Tariff.apply()` or `Contract.calculate_cost()` — a `timestamp` column and a 3-level `MultiIndex` on the cost columns — and returns the tax amounts:

In [2]:
from energy_cost import Meter, Tariff, TariffCategory

tariff = Tariff.from_yaml("../examples/tariffs/fixed.yml")

start = dt.datetime(2026, 1, 1, tzinfo=dt.UTC)
end = dt.datetime(2026, 4, 1, tzinfo=dt.UTC)

consumption = Meter(
    data=pd.DataFrame(
        {
            "timestamp": pd.date_range("2026-01-01T00:00:00+01:00", "2026-04-01T00:00:00+01:00", freq="15min"),
            "value": 0.0002,
        }
    )
)

cost_data = tariff.apply([consumption], start, end, tariff_category=TariffCategory.SUPPLIER)
cost_data

timestamp    supplier                           total
                            consumption     fixed         total   total
                                  total fixed_fee total   total   total
0 2026-01-01 00:00:00+00:00      71.424      15.0  15.0  86.424  86.424
1 2026-02-01 00:00:00+00:00      64.512      15.0  15.0  79.512  79.512
2 2026-03-01 00:00:00+00:00      71.352      15.0  15.0  86.352  86.352

In [ ]:
assert cost_data is not None
tax.apply(cost_data)

,timestamp,"(taxes, total, total)"
0,2026-01-01 00:00:00+00:00,18.14904
1,2026-02-01 00:00:00+00:00,16.69752
2,2026-03-01 00:00:00+00:00,18.13392


### 2. Defining a tax in YAML

Just like tariffs, taxes are typically defined in YAML files and loaded with `Tax.from_yaml()`.

In [4]:
from helpers import display_yaml  # ty: ignore[unresolved-import]

display_yaml("../examples/taxes/foo.yml")

```yaml
- start: 2022-01-01T00:00:00+01:00
  default: 0.06
  rates:
    - rate: 0.21
      columns:
        - ["*", "capacity", "*"]
    - rate: 0.0
      columns:
        - ["*", "injection", "*"]

```

In [5]:
foo = Tax.from_yaml("../examples/taxes/foo.yml")
foo

Tax(versions=[TaxVersion(start=datetime.datetime(2022, 1, 1, 0, 0, tzinfo=datetime.timezone(datetime.timedelta(seconds=3600))), default=0.06, rates=[TaxRule(rate=0.21, columns=[('*', <CostGroup.CAPACITY: 'capacity'>, '*')]), TaxRule(rate=0.0, columns=[('*', <CostGroup.INJECTION: 'injection'>, '*')])])])

### 3. Column-specific rates

Each rate targets cost columns using a 3-part pattern: `(category, cost_group, cost_type)` where `"*"` matches anything.

| Part | Possible values |
|------|-----------------|
| `category` | `supplier`, `distributor`, `fees`, `taxes`, `total` |
| `cost_group` | `consumption`, `injection`, `capacity`, `fixed`, `total` |
| `cost_type` | any string or `total` |

When multiple patterns match a column, the **most specific** one wins — specificity is determined by how many positions are non-wildcard, with the rightmost position weighted most. The `default` rate acts as the global fallback.

In the Foo example above:
- capacity costs are taxed at **21%** — matched by `["*", "capacity", "*"]`
- injection costs are **exempt** — matched by `["*", "injection", "*"]` with rate `0.0`
- everything else falls through to the **6%** default

To see this in action, let's apply it to cost data that includes consumption, injection, and capacity columns:

In [6]:
timestamps = pd.date_range("2025-01-01", periods=2, freq="MS", tz="UTC")

cols = pd.MultiIndex.from_tuples(
    [
        ("supplier", "consumption", "energy"),
        ("supplier", "consumption", "total"),
        ("distributor", "capacity", "peak_demand"),
        ("distributor", "capacity", "total"),
        ("supplier", "injection", "rebate"),
        ("supplier", "injection", "total"),
        ("supplier", "total", "total"),
        ("distributor", "total", "total"),
        ("total", "total", "total"),
    ]
)

values = [
    [100.0, 100.0, 50.0, 50.0, -20.0, -20.0, 80.0, 50.0, 130.0],
    [200.0, 200.0, 100.0, 100.0, -40.0, -40.0, 160.0, 100.0, 260.0],
]

mixed_costs = pd.DataFrame(values, columns=cols)
mixed_costs.insert(0, "timestamp", timestamps)
mixed_costs

timestamp    supplier        distributor         supplier  \
                            consumption           capacity        injection   
                                 energy  total peak_demand  total    rebate   
0 2025-01-01 00:00:00+00:00       100.0  100.0        50.0   50.0     -20.0   
1 2025-02-01 00:00:00+00:00       200.0  200.0       100.0  100.0     -40.0   

               distributor  total  
         total       total  total  
  total  total       total  total  
0 -20.0   80.0        50.0  130.0  
1 -40.0  160.0       100.0  260.0

In [7]:
foo.apply(mixed_costs)

,timestamp,"(taxes, total, total)"
0,2025-01-01 00:00:00+00:00,16.5
1,2025-02-01 00:00:00+00:00,33.0


As expected:
- capacity (`50`) is taxed at **21%** → `10.5`
- injection (`-20`) is **exempt** → `0`
- consumption (`100`) falls through to the **6%** default → `6`
- total tax = `10.5 + 0 + 6 = 16.5` per period

### 4. Versioning

Like tariffs, a tax definition can span multiple versions — each with its own rates — to handle changes in legislation over time. A new version becomes active from its `start` timestamp.

In [ ]:
from energy_cost import CostGroup
from energy_cost.tax import TaxRule

tax_with_versions = Tax(
    versions=[
        TaxVersion(
            start=dt.datetime(2023, 1, 1, tzinfo=dt.UTC),
            default=0.21,  # before the energy crisis reduction
        ),
        TaxVersion(
            start=dt.datetime(2024, 1, 1, tzinfo=dt.UTC),
            default=0.06,  # reduced rate introduced
            rates=[
                TaxRule(rate=0.21, columns=[("*", CostGroup.CAPACITY, "*")]),
            ],
        ),
    ]
)

tax_with_versions

Tax(versions=[TaxVersion(start=datetime.datetime(2023, 1, 1, 0, 0, tzinfo=datetime.timezone.utc), default=0.21, rates=[]), TaxVersion(start=datetime.datetime(2024, 1, 1, 0, 0, tzinfo=datetime.timezone.utc), default=0.06, rates=[TaxRule(rate=0.21, columns=[('*', <CostGroup.CAPACITY: 'capacity'>, '*')])])])

When applied to data spanning multiple version boundaries, `Tax.apply()` automatically uses the correct version for each time segment.

> Taxes are typically passed directly to `Contract` which handles the application automatically. See the [contract documentation](./contract.ipynb) for a complete example.